# 6 — Temporal Difference (TD) Learning: TD(0), SARSA, Q-learning

**Cel:** Zrozumieć bootstrap i różnice:
- MC vs TD(0) (bias/variance),
- SARSA (on-policy) vs Q-learning (off-policy).

TODO: WSTAW PSEUDOKODY (Sutton): TD(0), SARSA, Q-learning.


## Plan (ok. 2–2.5h)

1) Setup + DP reference  
2) Ćwiczenie 1: TD(0) prediction (Vπ)  
3) Eksperyment A: MC vs TD — V(start) vs epizody (wiele seedów)  
4) Ćwiczenie 2: SARSA control  
5) Bonus: Q-learning (run & interpret)  
6) Eksperyment B: SARSA vs Q-learning (det i slippery) + wpływ ε/α  
7) Krótka sekcja: „Most do DQN/PPO” (wyłącznie koncepcyjnie)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from typing import Optional, Dict, List, Tuple

# ---------- Viz helpers (consistent with Ch04) ----------
def pretty_matrix_as_grid(v: np.ndarray, nrow: int, ncol: int, decimals: int = 2):
    grid = np.asarray(v, dtype=float).reshape(nrow, ncol)
    with np.printoptions(precision=decimals, suppress=True):
        print(grid)

def action_arrows(pi_det: np.ndarray, nrow: int, ncol: int, arrows: Dict[int, str]):
    out = []
    for r in range(nrow):
        row = []
        for c in range(ncol):
            s = r * ncol + c
            row.append(arrows.get(int(pi_det[s]), '?'))
        out.append(' '.join(row))
    print('\n'.join(out))

def moving_average(x, window: int = 200):
    x = np.asarray(x, dtype=float)
    if len(x) < window:
        return x
    w = np.ones(window) / window
    return np.convolve(x, w, mode="valid")

# ---------- Environment wrapper: sample transitions from P[s][a] ----------
class _Discrete:
    def __init__(self, n: int):
        self.n = int(n)

class PModelEnv:
    \"\"\"Minimal env wrapper around model P[s][a].\"\"\"
    def __init__(self, P, start_state: int = 0, seed: int = 0):
        self.P = P
        self.nS = len(P)
        s0 = next(iter(P))
        self.nA = len(P[s0])
        self.action_space = _Discrete(self.nA)
        self.observation_space = _Discrete(self.nS)
        self.start_state = int(start_state)
        self.rng = np.random.default_rng(seed)
        self.s = self.start_state

    def reset(self, seed: Optional[int] = None):
        if seed is not None:
            self.rng = np.random.default_rng(int(seed))
        self.s = self.start_state
        return int(self.s), {}

    def step(self, a: int):
        a = int(a)
        outcomes = self.P[self.s][a]  # [(p, s2, r, terminated), ...]
        ps = np.array([o[0] for o in outcomes], dtype=float)
        idx = int(self.rng.choice(len(outcomes), p=ps/ps.sum()))
        p, s2, r, terminated = outcomes[idx]
        self.s = int(s2)
        return int(s2), float(r), bool(terminated), False, {}

# ---------- FrozenLake model builder (no Gym) ----------
def build_frozenlake_P(desc, is_slippery: bool = False):
    \"\"\"FrozenLake in P[s][a] format. Actions: 0=L,1=D,2=R,3=U.\"\"\"
    desc = np.asarray([list(row) for row in desc], dtype="<U1")
    nrow, ncol = desc.shape
    nS, nA = nrow * ncol, 4

    LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
    moves = {LEFT:(0,-1), DOWN:(1,0), RIGHT:(0,1), UP:(-1,0)}

    def to_s(r,c): return r*ncol+c
    def step_from(r,c,a):
        dr,dc = moves[a]
        r2,c2 = r+dr, c+dc
        if r2<0 or r2>=nrow or c2<0 or c2>=ncol:
            r2,c2 = r,c
        return r2,c2

    P = {s: {a: [] for a in range(nA)} for s in range(nS)}
    for r in range(nrow):
        for c in range(ncol):
            s = to_s(r,c)
            tile = desc[r,c]
            if tile in ("H","G"):
                for a in range(nA):
                    P[s][a] = [(1.0, s, 0.0, True)]
                continue

            for a in range(nA):
                if is_slippery:
                    candidates = [(a-1)%4, a, (a+1)%4]
                    probs = [1/3, 1/3, 1/3]
                else:
                    candidates = [a]
                    probs = [1.0]

                outcomes = []
                for a_real, p in zip(candidates, probs):
                    r2,c2 = step_from(r,c,a_real)
                    s2 = to_s(r2,c2)
                    tile2 = desc[r2,c2]
                    terminated = tile2 in ("H","G")
                    reward = 1.0 if tile2 == "G" else 0.0
                    outcomes.append((float(p), int(s2), float(reward), bool(terminated)))

                merged = {}
                for p, s2, rwd, term in outcomes:
                    key = (s2, rwd, term)
                    merged[key] = merged.get(key, 0.0) + p
                P[s][a] = [(p, s2, rwd, term) for (s2, rwd, term), p in merged.items()]
    return P, nS, nA, nrow, ncol, desc

# ---------- Shared env instances ----------
desc4 = ["SFFF","FHFH","FFFH","HFFG"]
P_fl_det, nS, nA, nrow, ncol, _ = build_frozenlake_P(desc4, is_slippery=False)
P_fl_slip, _, _, _, _, _ = build_frozenlake_P(desc4, is_slippery=True)

env = PModelEnv(P_fl_det, start_state=0, seed=0)
env_slip = PModelEnv(P_fl_slip, start_state=0, seed=0)

arrows_fl = {0:"←",1:"↓",2:"→",3:"↑"}

print("FrozenLake map:")
print("\\n".join(desc4))
print("n_states =", nS, "n_actions =", nA)


In [ ]:
# ---------- DP reference (Value Iteration) ----------
def greedy_policy_from_v(P, v: np.ndarray, gamma: float = 0.99) -> np.ndarray:
    nS = v.shape[0]
    s0 = next(iter(P))
    nA = len(P[s0])
    pi_det = np.zeros(nS, dtype=int)
    for s in range(nS):
        best_a = 0
        best_q = -np.inf
        for a in range(nA):
            q = 0.0
            for (p, s2, r, terminated) in P[s][a]:
                if terminated:
                    q += float(p) * float(r)
                else:
                    q += float(p) * (float(r) + gamma * float(v[int(s2)]))
            if q > best_q:
                best_q = q
                best_a = a
        pi_det[s] = best_a
    return pi_det

def value_iteration(P, gamma: float = 0.99, theta: float = 1e-10, max_iters: int = 100_000):
    nS = len(P)
    s0 = next(iter(P))
    nA = len(P[s0])
    v = np.zeros(nS, dtype=float)
    for _ in range(max_iters):
        delta = 0.0
        for s in range(nS):
            v_old = v[s]
            best_q = -np.inf
            for a in range(nA):
                q = 0.0
                for (p, s2, r, terminated) in P[s][a]:
                    if terminated:
                        q += float(p) * float(r)
                    else:
                        q += float(p) * (float(r) + gamma * float(v[int(s2)]))
                best_q = max(best_q, q)
            v[s] = best_q
            delta = max(delta, abs(v_old - v[s]))
        if delta < theta:
            break
    pi_det = greedy_policy_from_v(P, v, gamma=gamma)
    return pi_det, v

gamma = 0.99
pi_star_det, v_star_det = value_iteration(P_fl_det, gamma=gamma)
pi_star_slip, v_star_slip = value_iteration(P_fl_slip, gamma=gamma)

print("DP reference: v*(start) det =", float(v_star_det[0]), "| slip =", float(v_star_slip[0]))


In [ ]:
# ---------- Common model-free helpers ----------
def epsilon_greedy_action(q_s: np.ndarray, eps: float, rng: np.random.Generator) -> int:
    if rng.random() < eps:
        return int(rng.integers(0, len(q_s)))
    return int(np.argmax(q_s))

def generate_episode_det_policy(env: PModelEnv, pi_det: np.ndarray,
                                max_steps: int = 200, seed: Optional[int] = None):
    s, _ = env.reset(seed=seed)
    episode = []
    for _ in range(max_steps):
        a = int(pi_det[s])
        s2, r, done, trunc, _ = env.step(a)
        episode.append((s, a, r))
        if done or trunc:
            break
        s = s2
    return episode

def generate_episode_eps_greedy(env: PModelEnv, Q: np.ndarray, eps: float,
                                rng: np.random.Generator, max_steps: int = 200):
    s, _ = env.reset()
    episode = []
    for _ in range(max_steps):
        a = epsilon_greedy_action(Q[s], eps, rng)
        s2, r, done, trunc, _ = env.step(a)
        episode.append((s, a, r))
        if done or trunc:
            break
        s = s2
    return episode


---

## 1) TD(0) prediction

TD(0) robi update po *jednym kroku* (bootstrap):

$
V(s) \leftarrow V(s) + lpha ig(r + \gamma V(s') - V(s)ig)
$

To zmniejsza wariancję, ale wprowadza bias (bo używamy przybliżonego V).

### Ćwiczenie 1 — implementacja TD(0) prediction


In [ ]:
def td0_prediction(env: PModelEnv, pi_det: np.ndarray,
                   alpha: float = 0.1, gamma: float = 0.99,
                   episodes: int = 20_000, seed: int = 0, max_steps: int = 200):
    # TODO (STUDENT): Zaimplementuj TD(0) prediction.
    # - V = zeros(nS)
    # - for episode:
    #   s=reset; loop steps:
    #     a=pi_det[s]; step -> s2,r,done
    #     target = r if done else r + gamma*V[s2]
    #     V[s] += alpha*(target - V[s])
    #     stop if done
    nS = env.observation_space.n
    V = np.zeros(nS, dtype=float)
    rng = np.random.default_rng(seed)

    for _ in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 10_000_000)))
        for _ in range(max_steps):
            a = int(pi_det[s])
            s2, r, done, trunc, _ = env.step(a)
            target = float(r) if (done or trunc) else float(r) + gamma * float(V[s2])
            V[s] += alpha * (target - V[s])
            s = s2
            if done or trunc:
                break
    return V


### Demo: TD(0) vs DP (policy pi* det)


In [ ]:
V_td = td0_prediction(env, pi_star_det, alpha=0.1, gamma=0.99, episodes=20_000, seed=0)

print("TD V_pi (pi* det):")
pretty_matrix_as_grid(V_td, nrow, ncol, decimals=2)

print("\nDP V* (det):")
pretty_matrix_as_grid(v_star_det, nrow, ncol, decimals=2)

print("\nV(start): TD =", float(V_td[0]), "| DP =", float(v_star_det[0]))


---

## Eksperyment A (run & interpret): MC vs TD — V(start) vs epizody, wiele seedów

Cel: pokazać różnicę bias/variance na tej samej polityce (np. `pi_star_det`).

Zadanie:
- policz `V(start)` dla rosnącej liczby epizodów dla MC i TD,
- powtórz dla kilku seedów,
- narysuj 2 wykresy (MC i TD) albo jeden z dwoma krzywymi.


In [ ]:
def mc_prediction_first_visit(env: PModelEnv, pi_det: np.ndarray,
                              gamma: float = 0.99, episodes: int = 10_000,
                              seed: int = 0, max_steps: int = 200):
    # Reuse from Lab05 (here included for self-containment).
    nS = env.observation_space.n
    V = np.zeros(nS, dtype=float)
    N = np.zeros(nS, dtype=int)
    rng = np.random.default_rng(seed)
    for _ in range(episodes):
        ep_seed = int(rng.integers(0, 10_000_000))
        ep = generate_episode_det_policy(env, pi_det, max_steps=max_steps, seed=ep_seed)
        G = 0.0
        visited = set()
        for t in reversed(range(len(ep))):
            s, a, r = ep[t]
            G = float(r) + gamma * G
            if s in visited:
                continue
            visited.add(s)
            N[s] += 1
            V[s] += (G - V[s]) / N[s]
    return V

episode_list = [200, 500, 1_000, 2_000, 5_000, 10_000]
seeds = [0, 1, 2, 3, 4]
pi_ref = pi_star_det

mc_vals = {sd: [] for sd in seeds}
td_vals = {sd: [] for sd in seeds}

for sd in seeds:
    for E in episode_list:
        Vmc = mc_prediction_first_visit(env, pi_ref, gamma=0.99, episodes=E, seed=sd)
        Vtd = td0_prediction(env, pi_ref, alpha=0.1, gamma=0.99, episodes=E, seed=sd)
        mc_vals[sd].append(float(Vmc[0]))
        td_vals[sd].append(float(Vtd[0]))

plt.figure()
for sd in seeds:
    plt.plot(episode_list, mc_vals[sd], marker="o", alpha=0.8)
plt.axhline(float(v_star_det[0]), linestyle="--")
plt.xlabel("epizody")
plt.ylabel("V(start) MC")
plt.title("MC prediction: V(start) vs epizody (wiele seedów)")
plt.grid(True)
plt.show()

plt.figure()
for sd in seeds:
    plt.plot(episode_list, td_vals[sd], marker="o", alpha=0.8)
plt.axhline(float(v_star_det[0]), linestyle="--")
plt.xlabel("epizody")
plt.ylabel("V(start) TD(0)")
plt.title("TD(0) prediction: V(start) vs epizody (wiele seedów)")
plt.grid(True)
plt.show()

print("DP V*(start) =", float(v_star_det[0]))


---

## 2) TD control: SARSA (on-policy) i Q-learning (off-policy)

### SARSA (on-policy)
Aktualizacja:
- wybieramy `a` ε-greedy,
- przechodzimy do `s2`,
- wybieramy `a2` ε-greedy,
- update: `Q[s,a] <- Q[s,a] + α (r + γ Q[s2,a2] - Q[s,a])`.

### Q-learning (off-policy)
Update:
- wybieramy `a` ε-greedy (behavior),
- target używa `max_a' Q[s2,a']` (target policy greedy).

### Ćwiczenie 2 — implementacja SARSA


In [ ]:
def sarsa(env: PModelEnv,
          alpha: float = 0.5, gamma: float = 0.99,
          eps: float = 0.2, episodes: int = 20_000,
          seed: int = 0, max_steps: int = 200):
    # TODO (STUDENT): Zaimplementuj SARSA (tablicowe).
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=float)
    rng = np.random.default_rng(seed)
    success_curve = []

    for _ in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 10_000_000)))
        a = epsilon_greedy_action(Q[s], eps, rng)
        success = 0

        for _ in range(max_steps):
            s2, r, done, trunc, _ = env.step(a)
            if r > 0:
                success = 1

            if done or trunc:
                target = float(r)
                Q[s, a] += alpha * (target - Q[s, a])
                break

            a2 = epsilon_greedy_action(Q[s2], eps, rng)
            target = float(r) + gamma * float(Q[s2, a2])
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = s2, a2

        success_curve.append(success)

    pi_det = np.argmax(Q, axis=1).astype(int)
    return {"Q": Q, "pi": pi_det, "success_curve": np.asarray(success_curve, dtype=int)}


### Bonus: Q-learning (run & interpret)

Poniżej jest gotowa implementacja. Studenci mają:
- uruchomić,
- porównać z SARSA,
- napisać wnioski (on-policy vs off-policy).


In [ ]:
def q_learning(env: PModelEnv,
               alpha: float = 0.5, gamma: float = 0.99,
               eps: float = 0.2, episodes: int = 20_000,
               seed: int = 0, max_steps: int = 200):
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=float)
    rng = np.random.default_rng(seed)
    success_curve = []

    for _ in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(0, 10_000_000)))
        success = 0
        for _ in range(max_steps):
            a = epsilon_greedy_action(Q[s], eps, rng)
            s2, r, done, trunc, _ = env.step(a)
            if r > 0:
                success = 1
            target = float(r) if (done or trunc) else float(r) + gamma * float(np.max(Q[s2]))
            Q[s, a] += alpha * (target - Q[s, a])
            s = s2
            if done or trunc:
                break
        success_curve.append(success)

    pi_det = np.argmax(Q, axis=1).astype(int)
    return {"Q": Q, "pi": pi_det, "success_curve": np.asarray(success_curve, dtype=int)}


### Demo: SARSA vs Q-learning (det i slippery)


In [ ]:
out_sarsa_det = sarsa(env, episodes=25_000, eps=0.2, alpha=0.5, seed=0)
out_ql_det = q_learning(env, episodes=25_000, eps=0.2, alpha=0.5, seed=1)

out_sarsa_slip = sarsa(env_slip, episodes=80_000, eps=0.2, alpha=0.5, seed=2)
out_ql_slip = q_learning(env_slip, episodes=80_000, eps=0.2, alpha=0.5, seed=3)

plt.figure()
plt.plot(moving_average(out_sarsa_det["success_curve"], 500), label="SARSA det")
plt.plot(moving_average(out_ql_det["success_curve"], 500), label="Q-learn det")
plt.title("TD control (det)")
plt.xlabel("epizod (wygładzone 500)"); plt.ylabel("success rate")
plt.grid(True); plt.legend(); plt.show()

plt.figure()
plt.plot(moving_average(out_sarsa_slip["success_curve"], 1000), label="SARSA slip")
plt.plot(moving_average(out_ql_slip["success_curve"], 1000), label="Q-learn slip")
plt.title("TD control (slippery)")
plt.xlabel("epizod (wygładzone 1000)"); plt.ylabel("success rate")
plt.grid(True); plt.legend(); plt.show()

print("SARSA policy (slip):")
action_arrows(out_sarsa_slip["pi"], nrow, ncol, arrows_fl)
print("\nQ-learning policy (slip):")
action_arrows(out_ql_slip["pi"], nrow, ncol, arrows_fl)


---

## Eksperyment B (run & interpret): wpływ ε i α

Zadanie:
- zrób sweep po ε (np. 0.05, 0.1, 0.2, 0.3),
- (opcjonalnie) sweep po α (np. 0.1, 0.3, 0.5),
- mierz końcowy success rate (tail average) dla SARSA i Q-learning na slippery.

Wnioski: 6–10 zdań.


In [ ]:
eps_list = [0.05, 0.1, 0.2, 0.3]
alpha_list = [0.1, 0.3, 0.5]
episodes = 60_000

results = []
for eps in eps_list:
    for alpha in alpha_list:
        s_out = sarsa(env_slip, episodes=episodes, eps=eps, alpha=alpha, seed=0)
        q_out = q_learning(env_slip, episodes=episodes, eps=eps, alpha=alpha, seed=1)
        s_tail = float(np.mean(s_out["success_curve"][-10_000:]))
        q_tail = float(np.mean(q_out["success_curve"][-10_000:]))
        results.append((eps, alpha, s_tail, q_tail))

# Prosty wydruk tabeli
print("eps | alpha | SARSA_tail | QL_tail")
for row in results:
    print(f"{row[0]:.2f} | {row[1]:.1f} | {row[2]:.3f} | {row[3]:.3f}")


---

## Most do DQN / PPO (krótko, koncepcyjnie)

- Q-learning w tablicy nie skaluje się do dużych przestrzeni stanów.
- DQN: `Q(s,a)` aproksymujemy siecią, ale potrzebujemy stabilizacji:
  - replay buffer (mniejsza korelacja próbek),
  - target network (stabilny cel bootstrapu).

- PPO / Actor-Critic: zamiast wartości akcji w tablicy uczymy bezpośrednio politykę (stochastic policy) i wartość.

Oddanie (8–12 zdań):
1) MC vs TD (bias/variance) — w oparciu o eksperyment A  
2) SARSA vs Q-learning (on/off-policy) — szczególnie na slippery  
3) Jak ε/α wpływają na uczenie (z eksperymentu B)
